# Agentic Mobility Model - Complete Showcase

This notebook demonstrates the complete agentic mobility pipeline:
- **Natural Language → Mobility Simulation**: Convert text queries to UE mobility data
- **Topology Generation**: LLM-generated cell tower placement
- **Static & Interactive Visualizations**: Comprehensive analysis tools
- **Location Validation**: Reverse geocoding to verify spatial bounds

---

## Part 0: Environment Setup Instructions

### Step 1: Create Virtual Environment

```bash
# Navigate to project root
cd /path/to/maveric

python3 -m venv venv

source venv/bin/activate
```

### Step 2: Install Requirements

```bash
pip install -r radp/digital_twin/requirements.txt

pip install -r notebooks/requirements.txt

pip install -r requirements-dev.txt
```

### Step 3: Configure API Key

Set your LLM API key:

```bash
# Copy example environment file
cp radp/digital_twin/agentic_mobility/.env.example radp/digital_twin/agentic_mobility/.env

# Edit the .env file and add your API key and model
```

**Get Groq API Key:** https://console.groq.com/ (free tier available)

---

## Part 1: Introduction & Setup

### What is Agentic Mobility?

Agentic Mobility transforms RADP's mobility generator from **parameter-driven JSON** to **natural language queries** using LLM-powered workflows.

**Key Features:**
- **Natural Language Parsing**: "Generate 100 UEs in Tokyo" → valid parameters
- **Auto-Geocoding**: Location names → lat/lon bounds (cached)
- **Context-Aware**: LLM infers distributions from context (e.g., "rush hour" → high car %)
- **Self-Correction**: Auto-validates + retries with LLM suggestions
- **Parallel Execution**: Location + parameter generation run simultaneously
- **End-to-End**: Complete NL → DataFrame pipeline

### What This Notebook Demonstrates:

1. Natural language → mobility simulation
2. Cell tower topology generation
3. Static matplotlib visualizations
4. Scenario comparison
5. Location validation with reverse geocoding
6. Geographic world map visualization
7. Interactive Plotly dashboards
8. Export to CSV, JSON, PNG, HTML

---

In [ ]:
# Setup Python path
import sys
from pathlib import Path

# Add maveric root to path
notebook_dir = Path.cwd()
maveric_root = notebook_dir.parent
if str(maveric_root) not in sys.path:
    sys.path.insert(0, str(maveric_root))

print(f"Notebook directory: {notebook_dir}")
print(f"Maveric root: {maveric_root}")

In [ ]:
# Create output directory
output_dir = Path("data/agentic_data/mobility")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Output directory created: {output_dir}")
print(f"  Absolute path: {output_dir.resolve()}")
print(f"  Directory exists: {output_dir.exists()}")

In [ ]:
# Import required modules
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from radp.digital_twin.agentic_mobility.integration import AgenticMobilityIntegration
from radp.digital_twin.agentic_mobility.topology_generator import TopologyGenerator
from radp.digital_twin.agentic_mobility.visualization import (
    plot_ue_tracks,
    plot_ue_tracks_comparison,
    plot_ue_wise_interactive,
    plot_tick_wise_interactive,
    validate_location_bounds,
    plot_bounds_on_map,
)

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

print("All imports successful!")

---
## Part 2: Natural Language → Mobility Generation

This section demonstrates the complete pipeline from natural language query to mobility DataFrame.

The system will:
1. Parse the natural language query
2. Resolve the location to lat/lon bounds
3. Generate RADP-compatible parameters
4. Validate and self-correct if needed
5. Generate mobility simulation data

**Note**: First run may take ~5-10 seconds due to LLM API calls and geocoding.

---

In [ ]:
# Define natural language query
query = "Generate 100 UEs in urban Tokyo during morning rush hour"

print(f"Query: '{query}'")
print("\nProcessing...")
print("This may take 5-10 seconds (LLM + geocoding API calls)")

In [ ]:
# Generate mobility data from natural language
df, metadata = AgenticMobilityIntegration.generate_from_natural_language(query)

print(f"\nGenerated {len(df)} mobility points for {metadata['query_intent']['num_ues']} UEs")

In [ ]:
# Display key metadata
query_intent = metadata['query_intent']

print("="*70)
print("GENERATION METADATA")
print("="*70)
print(f"Location: {query_intent['location']}")
print(f"Scenario Type: {query_intent['scenario_type']}")
print(f"Number of UEs: {query_intent['num_ues']}")
print(f"Number of Ticks: {query_intent['num_ticks']}")
print(f"Retry Count: {metadata['retry_count']}")
print(f"\nUE Distribution (Source: {query_intent['ue_distribution']['source']}):")
for ue_type, percentage in query_intent['ue_distribution']['distribution'].items():
    print(f"  - {ue_type}: {percentage:.1%}")
print("="*70)

In [ ]:
# Display DataFrame preview
print("\nDataFrame Preview:")
display(df.head(10))

print(f"\nDataFrame Info:")
print(f"  - Total rows: {len(df)}")
print(f"  - Columns: {list(df.columns)}")
print(f"  - Unique UEs: {df['mock_ue_id'].nunique()}")
print(f"  - Ticks: {df['tick'].min()} to {df['tick'].max()}")
print(f"  - Lat range: [{df['lat'].min():.6f}, {df['lat'].max():.6f}]")
print(f"  - Lon range: [{df['lon'].min():.6f}, {df['lon'].max():.6f}]")

In [ ]:
# Save mobility data
csv_path = output_dir / "ue_tracks.csv"
metadata_path = output_dir / "ue_tracks_metadata.json"

df.to_csv(csv_path, index=False)
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved mobility data to: {csv_path}")
print(f"Saved metadata to: {metadata_path}")

---
## Part 3: Topology Generation

Generate cell tower topology using LLM-based intelligent placement.

The `TopologyGenerator` uses an LLM to:
- Understand the area type (urban/suburban/rural)
- Calculate optimal number of cell sites
- Place towers intelligently within spatial bounds
- Generate realistic cell configurations

---

In [ ]:
# Extract location data
location_data = metadata.get('location_data', {})
query_intent = metadata['query_intent']

# Generate topology using LLM
print("Generating cell tower topology using LLM...")
print("This may take a few seconds...\n")

topology_df = TopologyGenerator.generate_from_llm(
    area_type=location_data.get("area_type", "suburban"),
    num_ues=query_intent.get("num_ues", 100),
    min_lat=location_data.get("min_lat", df["lat"].min()),
    max_lat=location_data.get("max_lat", df["lat"].max()),
    min_lon=location_data.get("min_lon", df["lon"].min()),
    max_lon=location_data.get("max_lon", df["lon"].max()),
)

print(f"Generated {len(topology_df)} cell sectors")

In [ ]:
# Display topology preview
print("\nTopology Preview:")
display(topology_df.head(10))

# Count unique cell sites
unique_locations = topology_df.groupby(["cell_lat", "cell_lon"]).size()
print(f"\nGenerated {len(unique_locations)} unique cell sites")

In [ ]:
# Save topology
topology_path = output_dir / "cell_topology.csv"
topology_df.to_csv(topology_path, index=False)

# Save generation parameters
params_info = {
    "query": query,
    "location_data": location_data,
    "query_intent": query_intent,
    "metadata": metadata,
}
params_path = output_dir / "generation_params.json"
with open(params_path, 'w') as f:
    json.dump(params_info, f, indent=2)

print(f"Saved topology to: {topology_path}")
print(f"Saved generation parameters to: {params_path}")

In [ ]:
# Validate spatial boundaries
print("\nBoundary Validation:")
print("="*70)
print(f"  UE Lat range: [{df['lat'].min():.6f}, {df['lat'].max():.6f}]")
print(f"  UE Lon range: [{df['lon'].min():.6f}, {df['lon'].max():.6f}]")
print(f"  Cell Lat range: [{topology_df['cell_lat'].min():.6f}, {topology_df['cell_lat'].max():.6f}]")
print(f"  Cell Lon range: [{topology_df['cell_lon'].min():.6f}, {topology_df['cell_lon'].max():.6f}]")

# Check towers within UE bounds
towers_in_bounds = topology_df[
    (topology_df['cell_lat'] >= df['lat'].min()) & 
    (topology_df['cell_lat'] <= df['lat'].max()) &
    (topology_df['cell_lon'] >= df['lon'].min()) & 
    (topology_df['cell_lon'] <= df['lon'].max())
]
print(f"\n  Cell towers within UE bounds: {len(towers_in_bounds)}/{len(topology_df)}")
print("="*70)

---
## Part 4: Static Visualizations

Create static matplotlib visualizations of UE mobility tracks.

We'll create:
1. UE tracks with cell tower overlay
2. All UEs without legend (clean view)
3. Subset of UEs with legend (detailed view)

---

In [ ]:
# Plot UE tracks with cell towers
plt.figure(figsize=(12, 8))

# Plot UE tracks (first 20 UEs for clarity)
num_ues_to_plot = min(20, df['mock_ue_id'].nunique())
ue_ids = sorted(df['mock_ue_id'].unique())[:num_ues_to_plot]

for ue_id in ue_ids:
    ue_data = df[df['mock_ue_id'] == ue_id].sort_values('tick')
    plt.plot(ue_data['lon'], ue_data['lat'], alpha=0.6, linewidth=1)

# Plot cell towers
plt.scatter(
    topology_df['cell_lon'],
    topology_df['cell_lat'],
    c='red',
    marker='^',
    s=100,
    label='Cell Towers',
    zorder=5,
    edgecolors='darkred',
    linewidths=1
)

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'UE Mobility Tracks with Cell Tower Topology\n(Showing {num_ues_to_plot}/{df["mock_ue_id"].nunique()} UEs, {len(unique_locations)} Cell Sites)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save
viz_path = output_dir / "ue_tracks_with_topology.png"
plt.savefig(viz_path, dpi=150)
plt.show()

print(f"Saved visualization to: {viz_path}")

In [ ]:
# Plot all UEs without legend
print(f"Plotting all {df['mock_ue_id'].nunique()} UEs...")

plot_ue_tracks(
    df, 
    legend=False, 
    title=f"All {df['mock_ue_id'].nunique()} UEs - Complete Tracks"
)

# Save
viz_path = output_dir / "ue_tracks_all.png"
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f"Saved visualization to: {viz_path}")

In [ ]:
# Plot first 5 UEs with legend
ue_ids_subset = sorted(df['mock_ue_id'].unique())[:5]

plot_ue_tracks(
    df, 
    legend=True, 
    ue_ids=ue_ids_subset,
    show_end_points=True,
    title="First 5 UEs - Detailed View with Start/End Markers"
)

# Save
viz_path = output_dir / "ue_tracks_subset.png"
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f"Saved visualization to: {viz_path}")

---
## Part 5: Scenario Comparison

Generate a second scenario and compare side-by-side.

This demonstrates the flexibility of the agentic mobility system to handle different locations and scenarios.

---

In [ ]:
# Generate a different scenario
query2 = "Generate 50 UEs in suburban Austin, Texas with lots of cars"

print(f"Query 2: '{query2}'")
print("Processing...\n")

df2, metadata2 = AgenticMobilityIntegration.generate_from_natural_language(query2)

print(f"Generated {len(df2)} mobility points for {metadata2['query_intent']['num_ues']} UEs")

# Display key differences
print("\nComparison:")
print(f"  Scenario 1: {metadata['query_intent']['location']} ({metadata['query_intent']['scenario_type']})")
print(f"  Scenario 2: {metadata2['query_intent']['location']} ({metadata2['query_intent']['scenario_type']})")
print(f"  UEs: {df['mock_ue_id'].nunique()} vs {df2['mock_ue_id'].nunique()}")

In [ ]:
# Save scenario 2 data
csv_path2 = output_dir / "ue_tracks_scenario2.csv"
metadata_path2 = output_dir / "ue_tracks_scenario2_metadata.json"

df2.to_csv(csv_path2, index=False)
with open(metadata_path2, 'w') as f:
    json.dump(metadata2, f, indent=2)

print(f"Saved scenario 2 data to: {csv_path2}")
print(f"Saved scenario 2 metadata to: {metadata_path2}")

In [ ]:
# Side-by-side comparison
plot_ue_tracks_comparison(
    df, 
    df2,
    legend=False,
    titles=(
        f"Scenario 1: {metadata['query_intent']['location']} ({metadata['query_intent']['scenario_type']})",
        f"Scenario 2: {metadata2['query_intent']['location']} ({metadata2['query_intent']['scenario_type']})"
    )
)

# Save
viz_path = output_dir / "scenario_comparison.png"
plt.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f"Saved comparison to: {viz_path}")

---
## Part 6: Location Validation

Validate that the generated spatial bounds match the intended location using reverse geocoding.

The validation process:
1. Extracts 5 points: center + 4 corners (NW, NE, SW, SE)
2. Reverse geocodes each point using OpenStreetMap/Nominatim
3. Compares detected locations vs. query intent
4. Returns confidence scores and consistency metrics

**Note**: This makes 5 API calls with ~1 second rate limiting (~5 seconds total).

---

In [ ]:
# Run location validation
if metadata and 'spatial_bounds' in metadata:
    print("Performing location validation...")
    print(f"Query location: {metadata.get('query_intent', {}).get('location')}")
    print("\n" + "="*70)
    
    validation_result = validate_location_bounds(metadata, threshold=0.7)
    
    print("\nValidation Results:")
    print("="*70)
    
    if validation_result['is_match']:
        print("LOCATION VALIDATED")
    else:
        print("LOCATION MISMATCH")
    
    print(f"\nOverall Confidence: {validation_result['overall_confidence']:.1%}")
    print(f"Consistency Score: {validation_result['consistency_score']:.1%}")
    print(f"Query Location: {validation_result['query_location']}")
    
    print("\nDetected Locations (5 points):")
    print("-"*70)
    for point_name, location in validation_result['detected_locations'].items():
        confidence = validation_result['point_confidences'][point_name]
        city = location.get('city', 'Unknown')
        country = location.get('country', 'Unknown')
        print(f"  {point_name.upper():8s}: {city}, {country} (confidence: {confidence:.1%})")
    
    if validation_result['warnings']:
        print("\nWarnings:")
        for warning in validation_result['warnings']:
            print(f"  - {warning}")
    
    print("\n" + "="*70)
else:
    print("No metadata or spatial_bounds available for validation")
    validation_result = None

---
## Part 7: Geographic World Map Visualization

Visualize the spatial bounds on an interactive world map.

Features:
- Red rectangle showing spatial bounds
- 5 validation points with markers (blue center, green corners)
- Legend with full addresses
- Auto-zoom based on coordinate span
- Hover for detailed info

---

In [ ]:
# Create world map visualization
if metadata and 'spatial_bounds' in metadata and validation_result:
    print("Creating world map visualization...\n")
    
    fig_map = plot_bounds_on_map(
        validation_result,
        title="Spatial Bounds Validation - World Map View",
        figsize=(1200, 800)
    )
    
    fig_map.show()
    
    # Save to HTML
    map_path = output_dir / "bounds_world_map.html"
    fig_map.write_html(str(map_path))
    
    print(f"\nSaved interactive map to: {map_path}")
    print("\nInteractive Map Features:")
    print("   - Red rectangle shows the spatial bounds")
    print("   - Blue circle = Center point")
    print("   - Green diamonds = Corner points (NW, NE, SW, SE)")
    print("   - Hover over points to see full address details")
    print("   - Legend shows all 5 points with their geocoded addresses")
else:
    print("No validation result available for map visualization")

---
## Part 8: Interactive Plotly Visualizations

Create interactive Plotly dashboards for exploring the mobility data.

Two visualization modes:
1. **UE-wise**: Dropdown menu to select and view individual UE tracks
2. **Tick-wise**: Slider to animate through time and see all UEs at each tick

Both are exportable to standalone HTML files.

---

In [ ]:
# Create UE-wise interactive visualization
print("Creating UE-wise interactive visualization...\n")

ue_ids_subset = sorted(df['mock_ue_id'].unique())

fig_ue_wise = plot_ue_wise_interactive(
    df,
    ue_ids=ue_ids_subset,
    show_arrows=True,
    figsize=(1000, 700)
)

fig_ue_wise.show()

# Save to HTML
html_path = output_dir / "interactive_ue_wise.html"
fig_ue_wise.write_html(str(html_path))

print(f"Saved interactive UE-wise plot to: {html_path}")
print("\nFeatures:")
print("   - Use dropdown to switch between UEs")
print("   - Green circle = Start point")
print("   - Red square = End point")
print("   - Hover over points to see details (UE ID, tick, lat/lon)")

In [ ]:
# Create tick-wise interactive visualization
print("Creating tick-wise interactive visualization...\n")

fig_tick_wise = plot_tick_wise_interactive(
    df,
    initial_tick=0,
    color_by_ue=True,
    show_trails=False
)

fig_tick_wise.show()

# Save to HTML
html_path = output_dir / "interactive_tick_wise.html"
fig_tick_wise.write_html(str(html_path))

print(f"Saved interactive tick-wise plot to: {html_path}")
print("\nControls:")
print("   - Click PLAY button to animate through time")
print("   - Use slider to select specific tick")
print("   - Colors represent different UEs")
print("   - Hover to see UE ID, tick, and coordinates")

---
## Part 9: Summary & Generated Files

Review all generated files and key metrics from this demonstration.

---

In [ ]:
# List all generated files
print("="*70)
print("GENERATED FILES")
print("="*70)

all_files = sorted(output_dir.glob("*"))

for file_path in all_files:
    size_kb = file_path.stat().st_size / 1024
    print(f"{file_path.name:45s} {size_kb:>10.1f} KB")

print("\n" + "="*70)
print(f"Total files generated: {len(all_files)}")
print(f"Output directory: {output_dir.resolve()}")
print("="*70)

In [ ]:
# Display summary metrics
print("\nSUMMARY METRICS")
print("="*70)
print(f"Query: '{query}'")
print(f"Location: {metadata['query_intent']['location']}")
print(f"Scenario: {metadata['query_intent']['scenario_type']}")
print(f"Total UEs: {df['mock_ue_id'].nunique()}")
print(f"Total Ticks: {df['tick'].max() + 1}")
print(f"Total Mobility Points: {len(df)}")
print(f"Cell Towers Generated: {len(unique_locations)} sites ({len(topology_df)} sectors)")

if validation_result:
    status = "PASSED" if validation_result['is_match'] else "FAILED"
    print(f"Location Validation: {status}")
    print(f"Validation Confidence: {validation_result['overall_confidence']:.1%}")

print("="*70)

---
## Conclusion

This notebook demonstrated the complete agentic mobility pipeline:

**Key Achievements:**
1. Converted natural language to mobility simulation (~5-10 seconds)
2. Generated intelligent cell tower topology
3. Created comprehensive static visualizations
4. Validated location accuracy with reverse geocoding
5. Visualized bounds on world map
6. Built interactive dashboards for exploration
7. Exported all data and visualizations

**Next Steps:**
- Try different queries (urban/suburban/rural, different locations)
- Experiment with UE distributions
- Integrate with MRO optimization (see `agentic_mro.ipynb`)
- Export to RADP simulation pipeline

**All outputs saved to:** `notebooks/data/agentic_data/mobility/`

---